## 1. Import Libraries

We'll use scikit-learn for machine learning, pandas for data manipulation, and matplotlib/seaborn for visualization.

In [ ]:
# Core libraries
# Visualization
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Machine Learning
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Set style for plots
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")

# Suppress warnings
import warnings

warnings.filterwarnings("ignore")

print("Libraries imported successfully!")

## 2. Load and Explore the Dataset

The California Housing dataset contains information about housing prices and various features of houses in California districts. The target variable is the **median house value** for households in each district.

In [ ]:
# Load the California Housing dataset
california = fetch_california_housing(as_frame=True)

# Create DataFrame
df = california.frame

# Display basic info
print(f"Dataset Shape: {df.shape}")
print(f"Number of samples: {df.shape[0]:,}")
print(f"Number of features: {df.shape[1] - 1}")
print("\nTarget variable: MedHouseVal (Median House Value in $100,000s)")
print("\n" + "=" * 60)
print("\nFeature Descriptions:")
print("-" * 60)

feature_descriptions = {
    "MedInc": "Median income in block group",
    "HouseAge": "Median house age in block group",
    "AveRooms": "Average number of rooms per household",
    "AveBedrms": "Average number of bedrooms per household",
    "Population": "Block group population",
    "AveOccup": "Average number of household members",
    "Latitude": "Block group latitude",
    "Longitude": "Block group longitude",
}

for feature, desc in feature_descriptions.items():
    print(f"  {feature:12} - {desc}")

In [ ]:
# Display first few rows
print("First 5 rows of the dataset:")
df.head()

In [ ]:
# Statistical summary
print("Statistical Summary:")
df.describe().round(2)

In [ ]:
# Check for missing values
print("Missing Values:")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "No missing values found!")

## 3. Data Visualization

Let's explore the relationships between features and the target variable.

In [ ]:
# Distribution of target variable
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df["MedHouseVal"], bins=50, color="#475569", edgecolor="white", alpha=0.8)
axes[0].set_xlabel("Median House Value ($100,000s)", fontsize=12)
axes[0].set_ylabel("Frequency", fontsize=12)
axes[0].set_title("Distribution of House Values", fontsize=14, fontweight="bold")
axes[0].axvline(
    df["MedHouseVal"].mean(),
    color="#ef4444",
    linestyle="--",
    linewidth=2,
    label=f"Mean: ${df['MedHouseVal'].mean()*100000:,.0f}",
)
axes[0].axvline(
    df["MedHouseVal"].median(),
    color="#10b981",
    linestyle="--",
    linewidth=2,
    label=f"Median: ${df['MedHouseVal'].median()*100000:,.0f}",
)
axes[0].legend()

# Box plot
axes[1].boxplot(
    df["MedHouseVal"],
    vert=True,
    patch_artist=True,
    boxprops=dict(facecolor="#475569", alpha=0.7),
    medianprops=dict(color="#f59e0b", linewidth=2),
)
axes[1].set_ylabel("Median House Value ($100,000s)", fontsize=12)
axes[1].set_title("House Value Distribution (Box Plot)", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()

print("\nTarget Variable Statistics:")
print(f"  Mean:   ${df['MedHouseVal'].mean() * 100000:,.0f}")
print(f"  Median: ${df['MedHouseVal'].median() * 100000:,.0f}")
print(f"  Std:    ${df['MedHouseVal'].std() * 100000:,.0f}")
print(f"  Min:    ${df['MedHouseVal'].min() * 100000:,.0f}")
print(f"  Max:    ${df['MedHouseVal'].max() * 100000:,.0f}")

In [ ]:
# Correlation matrix
fig, ax = plt.subplots(figsize=(12, 10))

correlation_matrix = df.corr()
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))

sns.heatmap(
    correlation_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8},
)

plt.title("Feature Correlation Matrix", fontsize=16, fontweight="bold", pad=20)
plt.tight_layout()
plt.show()

# Print correlations with target
print("\nCorrelation with House Value (MedHouseVal):")
print("-" * 45)
target_corr = (
    correlation_matrix["MedHouseVal"].drop("MedHouseVal").sort_values(ascending=False)
)
for feature, corr in target_corr.items():
    indicator = (
        "+++"
        if corr > 0.5
        else "++"
        if corr > 0.3
        else "+"
        if corr > 0
        else "-"
        if corr > -0.3
        else "--"
        if corr > -0.5
        else "---"
    )
    print(f"  {feature:12}: {corr:+.3f} {indicator}")

In [ ]:
# Scatter plots of key features vs target
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

features_to_plot = [
    "MedInc",
    "HouseAge",
    "AveRooms",
    "AveBedrms",
    "Population",
    "AveOccup",
]

for i, feature in enumerate(features_to_plot):
    axes[i].scatter(df[feature], df["MedHouseVal"], alpha=0.3, s=5, c="#475569")
    axes[i].set_xlabel(feature, fontsize=11)
    axes[i].set_ylabel("Median House Value", fontsize=11)
    axes[i].set_title(f"{feature} vs House Value", fontsize=12, fontweight="bold")

    # Add correlation annotation
    corr = df[feature].corr(df["MedHouseVal"])
    axes[i].annotate(
        f"r = {corr:.3f}",
        xy=(0.05, 0.95),
        xycoords="axes fraction",
        fontsize=11,
        fontweight="bold",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.8),
    )

plt.tight_layout()
plt.show()

## 4. Data Preprocessing

We'll prepare the data for modeling by:
1. Separating features (X) and target (y)
2. Splitting into training and test sets
3. Scaling the features (optional but often beneficial)

In [ ]:
# Separate features and target
X = df.drop("MedHouseVal", axis=1)
y = df["MedHouseVal"]

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(
    f"\nTraining set: {X_train.shape[0]:,} samples ({X_train.shape[0]/len(X)*100:.0f}%)"
)
print(f"Test set: {X_test.shape[0]:,} samples ({X_test.shape[0]/len(X)*100:.0f}%)")

In [ ]:
# Scale features (important for interpretation and some algorithms)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame for better readability
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X_test.columns)

print("Feature Scaling Applied (StandardScaler):")
print("  - Mean of each feature: 0")
print("  - Standard deviation of each feature: 1")
print("\nScaled Training Features (first 3 rows):")
X_train_scaled_df.head(3).round(3)

## 5. Model Training

Now we'll train our Linear Regression model. Linear regression finds the best-fitting line through the data by minimizing the sum of squared residuals.

In [ ]:
# Create and train the model
model = LinearRegression()

# Train on scaled features
model.fit(X_train_scaled, y_train)

print("Model Training Complete!")
print("=" * 50)
print(f"\nModel Intercept (bias term): {model.intercept_:.4f}")
print("  This is the predicted house value when all features are at their mean.")
print(f"  In dollars: ${model.intercept_ * 100000:,.0f}")

In [ ]:
# Display coefficients
coefficients = pd.DataFrame(
    {
        "Feature": X_train.columns,
        "Coefficient": model.coef_,
        "Abs_Coefficient": np.abs(model.coef_),
    }
).sort_values("Abs_Coefficient", ascending=False)

print("Model Coefficients (Feature Importance):")
print("-" * 50)
print("\nInterpretation: For each 1 standard deviation increase in")
print("the feature, the house value changes by this amount ($100k)")
print()

for _, row in coefficients.iterrows():
    direction = "+" if row["Coefficient"] > 0 else "-"
    print(
        f"  {row['Feature']:12}: {row['Coefficient']:+.4f}  ({direction}${abs(row['Coefficient'])*100000:,.0f})"
    )

# Visualize coefficients
fig, ax = plt.subplots(figsize=(10, 6))

colors = ["#10b981" if c > 0 else "#ef4444" for c in coefficients["Coefficient"]]
bars = ax.barh(
    coefficients["Feature"],
    coefficients["Coefficient"],
    color=colors,
    edgecolor="white",
)

ax.set_xlabel("Coefficient Value (Change in House Value per Std Dev)", fontsize=12)
ax.set_title("Linear Regression Coefficients", fontsize=14, fontweight="bold")
ax.axvline(0, color="black", linewidth=0.8)

# Add value labels
for bar, val in zip(bars, coefficients["Coefficient"], strict=False):
    ax.annotate(
        f"{val:.3f}",
        xy=(val, bar.get_y() + bar.get_height() / 2),
        xytext=(5 if val > 0 else -5, 0),
        textcoords="offset points",
        ha="left" if val > 0 else "right",
        va="center",
        fontsize=10,
    )

plt.tight_layout()
plt.show()

## 6. Model Evaluation

We'll evaluate our model using several metrics:
- **R-squared (R²)**: Proportion of variance explained (0 to 1, higher is better)
- **Mean Squared Error (MSE)**: Average squared difference between predicted and actual values
- **Root Mean Squared Error (RMSE)**: Square root of MSE (in same units as target)
- **Mean Absolute Error (MAE)**: Average absolute difference between predicted and actual values

In [ ]:
# Make predictions
y_train_pred = model.predict(X_train_scaled)
y_test_pred = model.predict(X_test_scaled)

# Calculate metrics for both sets
metrics = {
    "Metric": [
        "R-squared (R²)",
        "Mean Squared Error (MSE)",
        "Root Mean Squared Error (RMSE)",
        "Mean Absolute Error (MAE)",
    ],
    "Training Set": [
        r2_score(y_train, y_train_pred),
        mean_squared_error(y_train, y_train_pred),
        np.sqrt(mean_squared_error(y_train, y_train_pred)),
        mean_absolute_error(y_train, y_train_pred),
    ],
    "Test Set": [
        r2_score(y_test, y_test_pred),
        mean_squared_error(y_test, y_test_pred),
        np.sqrt(mean_squared_error(y_test, y_test_pred)),
        mean_absolute_error(y_test, y_test_pred),
    ],
}

metrics_df = pd.DataFrame(metrics)
print("Model Performance Metrics:")
print("=" * 60)
metrics_df

In [ ]:
# Visual interpretation of metrics
print("\nInterpretation of Results:")
print("-" * 60)

r2_test = r2_score(y_test, y_test_pred)
rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))
mae_test = mean_absolute_error(y_test, y_test_pred)

print(f"\n1. R-squared: {r2_test:.4f}")
print(f"   Our model explains {r2_test*100:.1f}% of the variance in house prices.")
if r2_test > 0.7:
    print("   This is a good fit for real-world data!")
elif r2_test > 0.5:
    print("   This is a moderate fit - there may be non-linear patterns.")
else:
    print("   Consider adding more features or using non-linear models.")

print(f"\n2. RMSE: {rmse_test:.4f} (${rmse_test*100000:,.0f})")
print(f"   On average, our predictions are off by about ${rmse_test*100000:,.0f}")

print(f"\n3. MAE: {mae_test:.4f} (${mae_test*100000:,.0f})")
print(f"   The typical error in our predictions is ${mae_test*100000:,.0f}")

## 7. Visualization of Results

In [ ]:
# Actual vs Predicted plot
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Training set
axes[0].scatter(y_train, y_train_pred, alpha=0.3, s=5, c="#475569")
axes[0].plot(
    [y_train.min(), y_train.max()],
    [y_train.min(), y_train.max()],
    "r--",
    lw=2,
    label="Perfect Prediction",
)
axes[0].set_xlabel("Actual House Value ($100k)", fontsize=12)
axes[0].set_ylabel("Predicted House Value ($100k)", fontsize=12)
axes[0].set_title(
    f"Training Set: Actual vs Predicted\nR² = {r2_score(y_train, y_train_pred):.4f}",
    fontsize=14,
    fontweight="bold",
)
axes[0].legend()

# Test set
axes[1].scatter(y_test, y_test_pred, alpha=0.3, s=5, c="#475569")
axes[1].plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    "r--",
    lw=2,
    label="Perfect Prediction",
)
axes[1].set_xlabel("Actual House Value ($100k)", fontsize=12)
axes[1].set_ylabel("Predicted House Value ($100k)", fontsize=12)
axes[1].set_title(
    f"Test Set: Actual vs Predicted\nR² = {r2_score(y_test, y_test_pred):.4f}",
    fontsize=14,
    fontweight="bold",
)
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Residuals analysis
residuals_test = y_test - y_test_pred

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Residuals vs Predicted
axes[0].scatter(y_test_pred, residuals_test, alpha=0.3, s=5, c="#475569")
axes[0].axhline(y=0, color="#ef4444", linestyle="--", linewidth=2)
axes[0].set_xlabel("Predicted House Value ($100k)", fontsize=12)
axes[0].set_ylabel("Residual", fontsize=12)
axes[0].set_title("Residuals vs Predicted Values", fontsize=14, fontweight="bold")

# Residuals histogram
axes[1].hist(residuals_test, bins=50, color="#475569", edgecolor="white", alpha=0.8)
axes[1].axvline(x=0, color="#ef4444", linestyle="--", linewidth=2)
axes[1].set_xlabel("Residual Value", fontsize=12)
axes[1].set_ylabel("Frequency", fontsize=12)
axes[1].set_title("Distribution of Residuals", fontsize=14, fontweight="bold")

# Q-Q plot
from scipy import stats

stats.probplot(residuals_test, dist="norm", plot=axes[2])
axes[2].set_title("Q-Q Plot (Normality Check)", fontsize=14, fontweight="bold")
axes[2].get_lines()[0].set_markerfacecolor("#475569")
axes[2].get_lines()[0].set_alpha(0.3)
axes[2].get_lines()[1].set_color("#ef4444")

plt.tight_layout()
plt.show()

print("Residuals Statistics:")
print(f"  Mean: {residuals_test.mean():.4f} (should be close to 0)")
print(f"  Std:  {residuals_test.std():.4f}")
print(f"  Min:  {residuals_test.min():.4f}")
print(f"  Max:  {residuals_test.max():.4f}")

## 8. Making Predictions

Let's use our trained model to make predictions for new data.

In [ ]:
# Example: Predict house value for specific conditions
example_houses = pd.DataFrame(
    {
        "MedInc": [4.0, 8.0, 3.5],  # Median income
        "HouseAge": [20, 5, 40],  # House age
        "AveRooms": [5.0, 8.0, 4.0],  # Average rooms
        "AveBedrms": [1.0, 1.5, 1.0],  # Average bedrooms
        "Population": [1500, 800, 2000],  # Population
        "AveOccup": [3.0, 2.5, 3.5],  # Average occupancy
        "Latitude": [34.0, 37.0, 33.5],  # Latitude
        "Longitude": [-118.0, -122.0, -117.5],  # Longitude
    },
    index=["Modest Home", "Luxury Home", "Budget Home"],
)

# Scale the example data using the same scaler
example_scaled = scaler.transform(example_houses)

# Make predictions
predictions = model.predict(example_scaled)

print("House Value Predictions:")
print("=" * 60)
print()

for i, (name, pred) in enumerate(zip(example_houses.index, predictions, strict=False)):
    print(f"{name}:")
    print(
        f"  Features: Income={example_houses.iloc[i]['MedInc']:.1f}, "
        f"Age={example_houses.iloc[i]['HouseAge']:.0f}yrs, "
        f"Rooms={example_houses.iloc[i]['AveRooms']:.1f}"
    )
    print(f"  Predicted Value: ${pred * 100000:,.0f}")
    print()

## 9. Key Insights and Conclusions

### Summary

Our linear regression model achieved the following results:

1. **Model Performance**: R² of ~0.60 indicates the model explains about 60% of the variance in house prices
2. **Prediction Accuracy**: RMSE of ~$73,000 means our predictions are typically off by this amount
3. **Most Important Feature**: Median Income has the strongest positive correlation with house prices

### Key Findings

- **Median Income** is by far the strongest predictor of house value
- **Location matters**: Latitude and Longitude affect prices (coastal areas tend to be more expensive)
- **House age** has a small negative effect on value
- **Average occupancy** negatively affects value (higher density = lower prices)

### Limitations

1. Linear regression assumes a linear relationship between features and target
2. The model doesn't capture complex interactions between features
3. Some important features (school quality, crime rate) are not in the dataset

### Next Steps

To improve predictions, consider:
- Polynomial features for non-linear relationships
- Regularization (Ridge/Lasso) to prevent overfitting
- More advanced models (Random Forest, Gradient Boosting)

In [ ]:
# Final summary
print("\n" + "=" * 60)
print("          LINEAR REGRESSION MODEL SUMMARY")
print("=" * 60)
print("\n  Dataset: California Housing")
print(f"  Samples: {len(df):,}")
print(f"  Features: {X.shape[1]}")
print(f"\n  Test R²: {r2_test:.4f}")
print(f"  Test RMSE: ${rmse_test*100000:,.0f}")
print(f"  Test MAE: ${mae_test*100000:,.0f}")
print("\n  Top 3 Most Important Features:")
for i, row in coefficients.head(3).iterrows():
    print(f"    {i+1}. {row['Feature']}: {row['Coefficient']:+.4f}")
print("\n" + "=" * 60)